# Connect-sum RL experiments (persistent local setup)

This notebook is a persistent experiment variant of the dependency notebook. It is focused on connect-sum experiments and analysis, not dependency propagation.

## What this notebook does

1. Reuses a persistent workbook copy at `outputs/unknotting_connect_sum_experiments.xlsx` (created once).
2. Hardcodes base knots: trefoil `3_1` and both 5-crossing knots `5_1`, `5_2`.
3. Builds experiment families:
   - `3_1 # (all exact 10-crossing knots)`
   - `(5_1, 5_2) # (all exact 8-crossing knots)`
4. Flips one crossing, keeps configurable 13->13 cases, runs RL reduction, and identifies reduced knots in the workbook.
5. Writes per-trial JSONL + run summary JSON + run manifest JSON under `outputs/`.

## Start a run: local requirements

- Python 3.10+
- Virtual environment with `pip install -r requirements.txt`
- Jupyter Lab/Notebook
- `data/unknotting.xlsx` present
- Model file present in one of: `models/best_model.zip`, `models/ppo_knot_rl_spherogram_continued.zip`, `outputs/best_model.zip`

## Optional GCP setup (only if you use cloud training/data paths)

This repository can run fully local; GCP is optional.

If you do use GCP-backed paths, ensure:

- A Google Cloud project exists and billing is enabled.
- Cloud Storage API is enabled.
- You have bucket read access (typically Storage Object Viewer).
- `gcloud auth login` has been run for CLI workflows.
- `gcloud auth application-default login` has been run for ADC-based Python auth.
- `GOOGLE_CLOUD_PROJECT` is set when needed by your environment or helper code.


In [ ]:
# Local setup (run once per environment, if needed)
# %pip install -r requirements.txt

In [ ]:
import os, re, json, ast, math, random, csv, glob, time, shutil
from pathlib import Path
from dataclasses import dataclass
from fractions import Fraction
from collections import defaultdict
from typing import Iterable, Optional, List, Tuple, Dict, Any

import numpy as np
import pandas as pd

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

from tqdm.auto import tqdm

import snappy
from spherogram import Link

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# Local repository paths (no Google Drive needed)
from pathlib import Path
import sys

_candidate_roots = [Path.cwd(), Path.cwd().parent]
_repo_root = next(
    (candidate for candidate in _candidate_roots if (candidate / "src").exists()),
    Path.cwd(),
)
_src_dir = _repo_root / "src"
if _src_dir.exists() and str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from upperbounds.io.paths import prepare_notebook_paths

PATH_CONTEXT = prepare_notebook_paths(
    repo_root=_repo_root,
    experiment_workbook_name="unknotting_connect_sum_experiments.xlsx",
)
globals().update(PATH_CONTEXT.as_notebook_globals())
PATH_CONTEXT.print_summary()


In [ ]:
# ----------------------------
# Configuration
# ----------------------------

# Hardcoded experiment anchors
HARD_TREFOIL_KNOT_ID = "3_1"
HARD_FIVE_CROSSING_KNOT_IDS = ["5_1", "5_2"]

# Experiment families
RUN_TREFOIL_WITH_10_EXACT = True
RUN_FIVE_WITH_8_EXACT = True

# Crossing filters for the connect-sum study
REQUIRE_PREFLIP_13 = True
REQUIRE_POSTFLIP_13 = True

# Runtime limits
MAX_CASES_PER_FAMILY = None   # set int for smoke testing
MAX_FLIPS_PER_CASE = None     # set int to cap number of flipped crossings per case
NUM_VARIANTS_PER_KNOT = 1     # connect-sum trials use direct PD; keep 1 unless experimenting

# RL settings
UNKNOTTER_EPISODES_PER_FLIP = 1
UNKNOTTER_MAX_STEPS = 500
TRAIN_IF_MODEL_MISSING = True
TRAIN_STEPS_IF_NEEDED = 20000

# Keep original move settings available for helper compatibility
BACKTRACK_STEPS_MIN = 6
BACKTRACK_STEPS_MAX = 8
RIII_STEPS_MAX = 20

# Identification window
MIN_DATABASE_CROSSINGS = 1
MAX_DATABASE_CROSSINGS = 13

# Output behavior (analysis-first by default)
WRITE_UPDATED_WORKBOOK = False
SAVE_EVERY_KNOT = False
OVERWRITE_EXPERIMENT_OUTPUTS = True

# Dependency propagation controls
ENABLE_DEPENDENCY_PROPAGATION = True
MAX_PROPAGATION_PASSES = None
OVERWRITE_DEPENDENCY_OUTPUTS = True
MAX_DEPENDENCY_EVIDENCE_PER_KNOT = 5

RUN_STAMP = time.strftime("%Y%m%d-%H%M%S")
RESULTS_JSONL_PATH = OUT_DIR / "connect_sum_experiment_results.jsonl"
SUMMARY_JSON_PATH = OUT_DIR / "connect_sum_experiment_summary.json"
RUN_MANIFEST_PATH = OUT_DIR / "connect_sum_experiment_manifest.json"
ALL_DEPENDENCIES_JSON_PATH = OUT_DIR / "connect_sum_dependency_all.json"
UNIMPROVED_DEPENDENCIES_JSON_PATH = OUT_DIR / "connect_sum_dependency_unimproved.json"
PROPAGATION_RESULTS_JSON_PATH = OUT_DIR / "connect_sum_dependency_propagation_results.json"

MODEL_PATH_CANDIDATES = [
    BASE / "best_model.zip",
    BASE / "ppo_knot_rl_spherogram_continued.zip",
    OUT_DIR / "best_model.zip",
]

# Training data sources from the original notebook / paper pipeline
GCS_CSV_PATH_MAIN = "gs://gdm-unknotting/hard_unknots.csv"
GCS_CSV_PATH_VERY = "gs://gdm-unknotting/very_hard_unknots.csv"

# Optional local extras: only used if present
LOCAL_EXTRA_FILES = [
    BASE / "random_diagrams.csv",
    BASE / "random_diagrams.txt",
    BASE / "hard_unknots.csv",
    BASE / "very_hard_unknots.csv",
]

# Parallelization settings kept for compatibility
ENABLE_PARALLEL = False
PARALLEL_MAX_WORKERS = None
PARALLEL_RESERVE_CORES = 1
PARALLEL_INFERENCE_DEVICE = "cpu"


In [ ]:
# ----------------------------
# Helpers: workbook / parsing
# ----------------------------
import sys
from pathlib import Path

try:
    _repo_root = REPO_ROOT
except NameError:
    _repo_root = Path.cwd()
_src_dir = _repo_root / "src"
if _src_dir.exists() and str(_src_dir) not in sys.path:
    sys.path.insert(0, str(_src_dir))

from upperbounds.data.parsing import (
    canon_coeff_key,
    canon_coeff_key_mirror,
    ensure_minmax_coeffs,
    format_unknotting,
    normalize_invariant_key_cell,
    parse_pd_cell,
    parse_unknotting_entry,
    parse_vector_cell,
    pick_first_existing,
    span_abs,
    strip_leading_trailing_zeros,
)


In [ ]:
# ----------------------------
# Load workbook and identify columns
# ----------------------------
from upperbounds.io.workbook import load_workbook_with_columns

df, WORKBOOK_COLUMNS = load_workbook_with_columns(XLSX_PATH)
globals().update(WORKBOOK_COLUMNS.as_notebook_globals())
WORKBOOK_COLUMNS.print_summary()

display(df.head())


In [ ]:
# ----------------------------
# Jones polynomial / invariant helpers
# ----------------------------
from upperbounds.invariants.jones import (
    bracket_from_pd,
    crossing_sign_pd,
    jones_string_from_pd,
    jones_vector_from_pd,
    parse_jones_string_to_dict,
    poly_add,
    poly_dict_to_knotinfo_vector,
    poly_monom,
    poly_mul,
    poly_scale,
)


In [ ]:
# ----------------------------
# Fill missing Jones vectors in the database itself if needed
# ----------------------------
missing_before = 0
filled = 0
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Filling missing Jones vectors"):
    vec = parse_vector_cell(row.get(jones_col))
    if ensure_minmax_coeffs(vec) is not None:
        continue

    missing_before += 1
    pd_list = parse_pd_cell(row.get(pd_col))
    if pd_list is None:
        errors.append((idx, row.get(knot_col), "missing/bad PD"))
        continue

    if len(pd_list) > 20:
        # the direct bracket expansion is exponential in crossing number
        # so we skip very large PDs here; the improvement pipeline only
        # needs Jones vectors for rows that are actually matched later.
        continue

    new_vec, err = jones_vector_from_pd(pd_list)
    if new_vec is not None:
        df.at[idx, jones_col] = str(new_vec)
        filled += 1
    else:
        errors.append((idx, row.get(knot_col), err))

print("Missing Jones entries before:", missing_before)
print("Filled directly from PD:", filled)
print("Unfilled / errors:", len(errors))
if errors[:10]:
    print("Sample errors:", errors[:10])



In [ ]:
# ----------------------------
# Build Jones lookup from the workbook
# The lookup stores every matching knot. The dependency pass below first
# takes the MAX inside one ambiguous Jones class, then the MIN across
# different dependency classes found by the search.
# ----------------------------
lookup = defaultdict(list)

for idx, row in df.iterrows():
    vec = parse_vector_cell(row.get(jones_col))
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        continue

    uk = parse_unknotting_entry(row.get(u_col))
    if uk["upper"] is None:
        continue

    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    rec = {
        "row_index": int(idx),
        "knot": row.get(knot_col),
        "upper": int(uk["upper"]),
        "lower": None if uk["lower"] is None else int(uk["lower"]),
        "mirror": False,
    }
    lookup[(sp, canon_coeff_key(coeffs))].append({**rec, "mirror": False})
    lookup[(sp, canon_coeff_key_mirror(coeffs))].append({**rec, "mirror": True})

print("Lookup keys:", len(lookup))



In [ ]:
# ----------------------------
# Training / RL utilities
# ----------------------------
from upperbounds.rl.runtime import (
    EnvCfg,
    SphKnotEnv,
    clean_pd_lines,
    crossings,
    default_local_training_files,
    default_model_path_candidates,
    is_trivial_zero,
    load_or_train_ppo_model,
    load_training_pd_lines as _load_training_pd_lines,
    make_sb3_load_custom_objects,
    make_single_env,
    parse_link_strict,
    read_first_col_local,
    riii_shuffle_only_link,
    run_unknotter_on_pd,
    workbook_pd_lines as _workbook_pd_lines,
)

MODEL_PATH_CANDIDATES = default_model_path_candidates(MODELS_DIR, OUT_DIR)
LOCAL_EXTRA_FILES = default_local_training_files(TRAINING_DIR, DATA_DIR)


def workbook_pd_lines(max_keep: int | None = None) -> list[str]:
    return _workbook_pd_lines(df, pd_col, max_keep=max_keep)


def load_training_pd_lines() -> list[str]:
    return _load_training_pd_lines(
        local_extra_files=LOCAL_EXTRA_FILES,
        df=df,
        pd_col=pd_col,
        seed=SEED,
    )


In [ ]:
# ----------------------------
# Load or train PPO model
# ----------------------------
cfg = EnvCfg(max_steps=UNKNOTTER_MAX_STEPS, allow_backtrack=True, seed=SEED)

if "MODEL_PATH_CANDIDATES" not in globals():
    MODEL_PATH_CANDIDATES = default_model_path_candidates(MODELS_DIR, OUT_DIR)
if "LOCAL_EXTRA_FILES" not in globals():
    LOCAL_EXTRA_FILES = default_local_training_files(TRAINING_DIR, DATA_DIR)

model, best_model_path = load_or_train_ppo_model(
    model_path_candidates=MODEL_PATH_CANDIDATES,
    train_if_model_missing=TRAIN_IF_MODEL_MISSING,
    train_steps_if_needed=TRAIN_STEPS_IF_NEEDED,
    cfg=cfg,
    local_extra_files=LOCAL_EXTRA_FILES,
    df=df,
    pd_col=pd_col,
    seed=SEED,
    output_model_path=OUT_DIR / "best_model.zip",
)
ACTIVE_MODEL_PATH = Path(best_model_path)
print("Active model path:", ACTIVE_MODEL_PATH)


In [ ]:
# ----------------------------
# Connect-sum, flip, and matching helpers
# ----------------------------
def flip_crossing_quad(quad):
    a, b, c, d = quad
    return [b, c, d, a]

def generate_single_flip_variants(pd_list):
    out = []
    n = len(pd_list)
    for i in range(n):
        flipped = []
        for j, quad in enumerate(pd_list):
            flipped.append(flip_crossing_quad(quad) if i == j else list(quad))
        out.append((i, flipped))
    return out

def pd_from_link(link_obj):
    return [list(q) for q in link_obj.PD_code()]

def connect_sum_pd(pd_left, pd_right):
    left = Link([list(q) for q in pd_left])
    right = Link([list(q) for q in pd_right])

    # Try several API shapes because spherogram versions differ.
    if hasattr(left, "connected_sum") and callable(getattr(left, "connected_sum")):
        out = left.connected_sum(right)
        if out is None:
            out = left
        return pd_from_link(out), None

    if hasattr(left, "connect_sum") and callable(getattr(left, "connect_sum")):
        out = left.connect_sum(right)
        if out is None:
            out = left
        return pd_from_link(out), None

    try:
        out = left + right
        return pd_from_link(out), None
    except Exception as e:
        return None, f"connect_sum_not_available: {e!r}"

def match_jones_vector_to_database(vec):
    parsed = ensure_minmax_coeffs(vec)
    if parsed is None:
        return []
    mn, mx, coeffs = parsed
    sp = span_abs(mn, mx)
    return lookup.get((sp, canon_coeff_key(coeffs)), [])

def best_upper_bound_from_matches(matches):
    if not matches:
        return None
    uppers = [m["upper"] for m in matches if m.get("upper") is not None]
    if not uppers:
        return None
    return max(uppers)

def normalize_knot_id(knot):
    return str(knot).strip()

def current_bounds_for_row(row_index):
    if row_index is None or row_index not in df.index:
        return {"kind": "missing", "lower": None, "upper": None, "raw": None}
    return parse_unknotting_entry(df.at[row_index, u_col])

def current_upper_for_match(match):
    row_index = match.get("row_index")
    bounds = current_bounds_for_row(row_index)
    if bounds["upper"] is not None:
        return int(bounds["upper"])
    if match.get("upper") is not None:
        return int(match["upper"])
    return None

def safe_dependency_matches_from_jones_matches(matches, source_knot=None):
    source_key = None if source_knot is None else normalize_knot_id(source_knot)
    usable = []
    for match in matches:
        knot_key = normalize_knot_id(match.get("knot"))
        if source_key is not None and knot_key == source_key:
            continue
        upper = current_upper_for_match(match)
        if upper is None:
            continue
        usable.append((int(upper), match))
    if not usable:
        return []
    safe_upper = max(upper for upper, _match in usable)
    return [match for upper, match in usable if upper == safe_upper]

def dependency_sort_key(dep):
    upper = dep.get("upper")
    return (10**9 if upper is None else int(upper), normalize_knot_id(dep.get("knot")))

def add_dependency(dependency_map, match, context, source_knot):
    dep_knot = normalize_knot_id(match.get("knot"))
    if dep_knot == normalize_knot_id(source_knot):
        return None

    row_index = match.get("row_index")
    bounds = current_bounds_for_row(row_index)
    upper = bounds.get("upper")
    lower = bounds.get("lower")
    if upper is None:
        upper = match.get("upper")
    if lower is None:
        lower = match.get("lower")
    if upper is None:
        return None

    rec = dependency_map.get(dep_knot)
    if rec is None:
        rec = {
            "knot": dep_knot,
            "row_index": None if row_index is None else int(row_index),
            "lower": None if lower is None else int(lower),
            "upper": int(upper),
            "raw_unknotting": None if bounds.get("raw") is None else str(bounds.get("raw")),
            "evidence_count": 0,
            "evidence": [],
        }
        dependency_map[dep_knot] = rec
    else:
        rec["lower"] = None if lower is None else int(lower)
        rec["upper"] = int(upper)
        rec["raw_unknotting"] = None if bounds.get("raw") is None else str(bounds.get("raw"))

    rec["evidence_count"] += 1
    if len(rec["evidence"]) < MAX_DEPENDENCY_EVIDENCE_PER_KNOT:
        ev = dict(context)
        ev["mirror"] = bool(match.get("mirror", False))
        rec["evidence"].append(ev)
    return rec


In [ ]:
# ----------------------------
# Build connect-sum source cohorts and experiment cases
# ----------------------------
_knot_order_pat = re.compile(r"^(\d+)(?:[a-zA-Z])?_(\d+)$")

def knot_crossing_and_number(knot_name):
    m = _knot_order_pat.match(str(knot_name).strip())
    if not m:
        return None
    return int(m.group(1)), int(m.group(2))

exact_10 = []
unknown_10 = []
exact_8 = []
five_rows = {}

for idx, row in df.iterrows():
    knot_name = normalize_knot_id(row.get(knot_col))
    order = knot_crossing_and_number(knot_name)
    if order is None:
        continue
    crossing_number, knot_num = order
    uk = parse_unknotting_entry(row.get(u_col))
    rec = {
        "row_index": int(idx),
        "knot": knot_name,
        "crossing": int(crossing_number),
        "knot_number": int(knot_num),
        "lower": uk.get("lower"),
        "upper": uk.get("upper"),
        "kind": uk.get("kind"),
    }
    if crossing_number == 10:
        if uk["kind"] == "exact":
            exact_10.append(rec)
        else:
            unknown_10.append(rec)
    if crossing_number == 8 and uk["kind"] == "exact":
        exact_8.append(rec)
    if crossing_number == 5 and knot_name in set(HARD_FIVE_CROSSING_KNOT_IDS):
        five_rows[knot_name] = rec

exact_10.sort(key=lambda r: r["knot_number"])
unknown_10.sort(key=lambda r: r["knot_number"])
exact_8.sort(key=lambda r: r["knot_number"])

missing_fives = [k for k in HARD_FIVE_CROSSING_KNOT_IDS if k not in five_rows]
if missing_fives:
    raise ValueError(f"Missing required 5-crossing knots in workbook: {missing_fives}")

trefoil_rows = [r for r in exact_10 + unknown_10 + exact_8 if r["knot"] == HARD_TREFOIL_KNOT_ID]
if not trefoil_rows:
    # Trefoil may not be in these three cohorts; locate globally.
    trefoil_idx = df.index[df[knot_col].astype(str).str.strip() == HARD_TREFOIL_KNOT_ID].tolist()
    if not trefoil_idx:
        raise ValueError(f"Trefoil knot {HARD_TREFOIL_KNOT_ID} not found in workbook")
    tidx = int(trefoil_idx[0])
    trefoil_row = {"row_index": tidx, "knot": HARD_TREFOIL_KNOT_ID}
else:
    trefoil_row = trefoil_rows[0]

print("10-crossing exact known:", len(exact_10))
print("10-crossing non-exact (unknown/range):", len(unknown_10))
print("8-crossing exact known:", len(exact_8))
print("5-crossing anchors:", sorted(five_rows))

experiment_cases = []
if RUN_TREFOIL_WITH_10_EXACT:
    for rec in exact_10:
        experiment_cases.append({
            "family": "trefoil_x_10_exact",
            "left_knot": HARD_TREFOIL_KNOT_ID,
            "right_knot": rec["knot"],
            "left_row_index": int(trefoil_row["row_index"]),
            "right_row_index": int(rec["row_index"]),
        })

if RUN_FIVE_WITH_8_EXACT:
    for five_id in HARD_FIVE_CROSSING_KNOT_IDS:
        for rec in exact_8:
            experiment_cases.append({
                "family": "five_x_8_exact",
                "left_knot": five_id,
                "right_knot": rec["knot"],
                "left_row_index": int(five_rows[five_id]["row_index"]),
                "right_row_index": int(rec["row_index"]),
            })

if MAX_CASES_PER_FAMILY is not None:
    by_family = defaultdict(list)
    for case in experiment_cases:
        by_family[case["family"]].append(case)
    trimmed = []
    for fam, fam_cases in by_family.items():
        trimmed.extend(fam_cases[:MAX_CASES_PER_FAMILY])
    experiment_cases = trimmed

print("Total experiment cases:", len(experiment_cases))
if experiment_cases:
    display(pd.DataFrame(experiment_cases).head(20))


In [ ]:
# ----------------------------
# Main connect-sum experiment loop (with dependency propagation)
# ----------------------------
def json_safe(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, Path):
        return str(obj)
    raise TypeError(f"Object of type {type(obj).__name__} is not JSON serializable")

def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2, default=json_safe)

def append_jsonl(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False, default=json_safe) + "\n")

if OVERWRITE_EXPERIMENT_OUTPUTS:
    cleanup_paths = [RESULTS_JSONL_PATH, SUMMARY_JSON_PATH]
    if OVERWRITE_DEPENDENCY_OUTPUTS:
        cleanup_paths.extend([
            ALL_DEPENDENCIES_JSON_PATH,
            UNIMPROVED_DEPENDENCIES_JSON_PATH,
            PROPAGATION_RESULTS_JSON_PATH,
        ])
    for path in cleanup_paths:
        p = Path(path)
        if p.exists():
            p.unlink()

manifest = {
    "created_at": RUN_STAMP,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "seed": int(SEED),
    "trefoil": HARD_TREFOIL_KNOT_ID,
    "five_crossing_knots": list(HARD_FIVE_CROSSING_KNOT_IDS),
    "require_preflip_13": bool(REQUIRE_PREFLIP_13),
    "require_postflip_13": bool(REQUIRE_POSTFLIP_13),
    "max_cases_per_family": MAX_CASES_PER_FAMILY,
    "max_flips_per_case": MAX_FLIPS_PER_CASE,
    "dependency_propagation_enabled": bool(ENABLE_DEPENDENCY_PROPAGATION),
    "propagation_max_passes": MAX_PROPAGATION_PASSES,
    "counts": {
        "exact_10": len(exact_10),
        "unknown_10": len(unknown_10),
        "exact_8": len(exact_8),
        "cases": len(experiment_cases),
    },
}
write_json(RUN_MANIFEST_PATH, manifest)

results = []
family_stats = defaultdict(lambda: {"cases": 0, "flips": 0, "kept_after_filter": 0, "matched": 0})

timestamp = RUN_STAMP
case_nodes_for_propagation = []

for cnum, case in enumerate(experiment_cases, start=1):
    family = case["family"]
    family_stats[family]["cases"] += 1

    source_case_key = f"case_{cnum}:{normalize_knot_id(case['left_knot'])}#{normalize_knot_id(case['right_knot'])}"
    left_pd = parse_pd_cell(df.at[case["left_row_index"], pd_col])
    right_pd = parse_pd_cell(df.at[case["right_row_index"], pd_col])
    if left_pd is None or right_pd is None:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "missing_pd",
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    sum_pd, cs_err = connect_sum_pd(left_pd, right_pd)
    if sum_pd is None:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "connect_sum_failed",
            "error": cs_err,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    preflip_crossings = len(sum_pd)
    if REQUIRE_PREFLIP_13 and preflip_crossings != 13:
        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "status": "filtered_preflip_crossings",
            "preflip_crossings": preflip_crossings,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)
        continue

    flip_variants = generate_single_flip_variants(sum_pd)
    if MAX_FLIPS_PER_CASE is not None:
        flip_variants = flip_variants[:MAX_FLIPS_PER_CASE]

    dependency_map = {}
    best_case_upper = None

    for flip_index, flipped_pd in flip_variants:
        family_stats[family]["flips"] += 1
        postflip_crossings = len(flipped_pd)
        if REQUIRE_POSTFLIP_13 and postflip_crossings != 13:
            continue

        family_stats[family]["kept_after_filter"] += 1
        success, min_crossings_found, best_pd = run_unknotter_on_pd(
            flipped_pd,
            model,
            cfg,
            episodes=UNKNOTTER_EPISODES_PER_FLIP,
            return_best_pd=True,
        )

        reduced_crossings = None if best_pd is None else len(best_pd)
        vec = None
        match_count = 0
        match_knots = []
        matched_upper = None
        candidate_upper = None
        status = "ran_no_match"

        if best_pd is not None and MIN_DATABASE_CROSSINGS <= reduced_crossings <= MAX_DATABASE_CROSSINGS:
            vec, err = jones_vector_from_pd(best_pd)
            if vec is not None:
                matches = match_jones_vector_to_database(vec)
                match_count = len(matches)
                if matches:
                    family_stats[family]["matched"] += 1
                    status = "matched"
                    match_knots = sorted({str(m.get("knot")) for m in matches})

                    context = {
                        "case_index": int(cnum),
                        "flip_index": int(flip_index),
                        "rl_success": bool(success),
                        "min_crossings_found": int(min_crossings_found),
                        "best_pd_crossings": int(reduced_crossings),
                        "jones_vector": vec,
                    }
                    safe_matches = safe_dependency_matches_from_jones_matches(matches, source_knot=source_case_key)
                    deps_for_this_run = []
                    for match in safe_matches:
                        rec = add_dependency(dependency_map, match, context, source_knot=source_case_key)
                        if rec is not None:
                            deps_for_this_run.append(rec)

                    if deps_for_this_run:
                        matched_upper = max(dep["upper"] for dep in deps_for_this_run if dep.get("upper") is not None)
                        candidate_upper = int(matched_upper) + 1
                        if best_case_upper is None or candidate_upper < int(best_case_upper):
                            best_case_upper = int(candidate_upper)

        row = {
            "case_index": cnum,
            "case_key": source_case_key,
            "family": family,
            "left_knot": case["left_knot"],
            "right_knot": case["right_knot"],
            "flip_index": int(flip_index),
            "status": status,
            "preflip_crossings": int(preflip_crossings),
            "postflip_crossings": int(postflip_crossings),
            "rl_success": bool(success),
            "min_crossings_found": int(min_crossings_found),
            "reduced_crossings": reduced_crossings,
            "jones_vector": vec,
            "match_count": int(match_count),
            "match_knots": match_knots,
            "matched_upper": matched_upper,
            "candidate_upper": candidate_upper,
        }
        results.append(row)
        append_jsonl(RESULTS_JSONL_PATH, row)

    dependencies = sorted(dependency_map.values(), key=dependency_sort_key)
    best_dependency_upper = min([d["upper"] for d in dependencies], default=None)
    dependency_candidate_upper = None if best_dependency_upper is None else int(best_dependency_upper) + 1

    case_nodes_for_propagation.append({
        "case_index": int(cnum),
        "case_key": source_case_key,
        "family": family,
        "left_knot": normalize_knot_id(case["left_knot"]),
        "right_knot": normalize_knot_id(case["right_knot"]),
        "direct_upper": best_case_upper,
        "current_upper": best_case_upper,
        "dependency_candidate_upper": dependency_candidate_upper,
        "dependencies": dependencies,
    })

all_payload = {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "results": results,
    "case_nodes": case_nodes_for_propagation,
}
write_json(ALL_DEPENDENCIES_JSON_PATH, all_payload)


# ----------------------------
# Dependency propagation over case nodes
# ----------------------------
def dependency_current_upper(dep, case_upper_by_key):
    dep_knot = normalize_knot_id(dep.get("knot"))
    if dep_knot in case_upper_by_key and case_upper_by_key[dep_knot] is not None:
        return int(case_upper_by_key[dep_knot])
    dep_upper = dep.get("upper")
    if dep_upper is None:
        return None
    return int(dep_upper)

def best_dependency_candidate(rec, case_upper_by_key):
    dep_bounds = []
    for dep in rec.get("dependencies", []):
        dep_upper = dependency_current_upper(dep, case_upper_by_key)
        if dep_upper is None:
            continue
        dep_bounds.append({
            "knot": normalize_knot_id(dep.get("knot")),
            "upper": int(dep_upper),
        })
    best_dep = min(dep_bounds, key=lambda d: (d["upper"], d["knot"]), default=None)
    candidate_upper = None if best_dep is None else int(best_dep["upper"]) + 1
    return candidate_upper, best_dep, sorted(dep_bounds, key=lambda d: (d["upper"], d["knot"]))

case_upper_by_key = {}
for rec in case_nodes_for_propagation:
    case_upper_by_key[rec["case_key"]] = rec.get("current_upper")

unimproved_case_nodes = [
    rec for rec in case_nodes_for_propagation
    if rec.get("direct_upper") is None and rec.get("dependency_candidate_upper") is not None
]

dependents_by_dependency = defaultdict(set)
for rec in case_nodes_for_propagation:
    src_case = rec.get("case_key")
    for dep in rec.get("dependencies", []):
        dep_knot = normalize_knot_id(dep.get("knot"))
        if dep_knot:
            dependents_by_dependency[dep_knot].add(src_case)

propagation_events = []
updated_by_propagation = defaultdict(list)
case_nodes_by_key = {rec["case_key"]: rec for rec in case_nodes_for_propagation}
pending = {rec["case_key"] for rec in case_nodes_for_propagation if rec.get("dependencies")}
pass_num = 0

if ENABLE_DEPENDENCY_PROPAGATION:
    while pending:
        if MAX_PROPAGATION_PASSES is not None and pass_num >= MAX_PROPAGATION_PASSES:
            break

        pass_num += 1
        current_batch = sorted(pending)
        pending = set()
        pass_updates = 0

        for case_key in current_batch:
            rec = case_nodes_by_key.get(case_key)
            if rec is None:
                continue

            current_upper = rec.get("current_upper")
            candidate_upper, best_dep, dep_bounds = best_dependency_candidate(rec, case_upper_by_key)
            should_update = (
                candidate_upper is not None and
                (current_upper is None or int(candidate_upper) < int(current_upper))
            )
            if not should_update:
                continue

            old_upper = current_upper
            rec["current_upper"] = int(candidate_upper)
            case_upper_by_key[case_key] = int(candidate_upper)
            pass_updates += 1

            event = {
                "pass": int(pass_num),
                "case_key": case_key,
                "case_index": int(rec["case_index"]),
                "old_upper": None if old_upper is None else int(old_upper),
                "new_upper": int(candidate_upper),
                "best_dependency": best_dep,
                "dependency_bounds_now": dep_bounds,
            }
            propagation_events.append(event)
            updated_by_propagation[case_key].append(event)

            for dependent in dependents_by_dependency.get(case_key, set()):
                if dependent != case_key:
                    pending.add(dependent)

        if pass_updates == 0:
            break

propagation_results = []
for rec in case_nodes_for_propagation:
    case_key = rec["case_key"]
    candidate_upper, best_dep, dep_bounds = best_dependency_candidate(rec, case_upper_by_key)
    propagation_results.append({
        "case_key": case_key,
        "case_index": int(rec["case_index"]),
        "family": rec["family"],
        "left_knot": rec["left_knot"],
        "right_knot": rec["right_knot"],
        "direct_upper": rec.get("direct_upper"),
        "final_upper": rec.get("current_upper"),
        "next_candidate_upper": candidate_upper,
        "updated_by_propagation": case_key in updated_by_propagation,
        "events": updated_by_propagation.get(case_key, []),
        "best_dependency_now": best_dep,
        "dependency_bounds_now": dep_bounds,
    })

write_json(UNIMPROVED_DEPENDENCIES_JSON_PATH, {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "nodes": unimproved_case_nodes,
})

write_json(PROPAGATION_RESULTS_JSON_PATH, {
    "created_at": timestamp,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "propagation_enabled": bool(ENABLE_DEPENDENCY_PROPAGATION),
    "propagation_max_passes": MAX_PROPAGATION_PASSES,
    "propagation_passes_run": int(pass_num),
    "propagation_updates": len(propagation_events),
    "events": propagation_events,
    "results": propagation_results,
})

summary = {
    "created_at": RUN_STAMP,
    "source_workbook": str(SOURCE_XLSX_PATH),
    "copied_workbook": str(XLSX_PATH),
    "result_count": len(results),
    "families": {k: v for k, v in family_stats.items()},
    "dependency_nodes": len(case_nodes_for_propagation),
    "propagation_updates": len(propagation_events),
    "propagation_passes_run": int(pass_num),
}
write_json(SUMMARY_JSON_PATH, summary)

print("Run manifest:", RUN_MANIFEST_PATH)
print("Per-trial JSONL:", RESULTS_JSONL_PATH)
print("Summary JSON:", SUMMARY_JSON_PATH)
print("All dependency JSON:", ALL_DEPENDENCIES_JSON_PATH)
print("Unimproved dependency JSON:", UNIMPROVED_DEPENDENCIES_JSON_PATH)
print("Propagation results JSON:", PROPAGATION_RESULTS_JSON_PATH)
display(pd.DataFrame(results).head(30))


Dependency propagation notebook for all configured targets.
